In [0]:
# Cell 1: Updated Setup
%pip install yfinance plotly lxml --quiet


Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
# Cell 2
dbutils.library.restartPython()

In [0]:
#Cell 3
import yfinance as yf
import pandas as pd
import warnings
import urllib.request

warnings.simplefilter(action='ignore', category=FutureWarning)

saudi_names = {
    "2222.SR": "Saudi Aramco", "1120.SR": "Al Rajhi Bank", "7010.SR": "stc",
    "2010.SR": "SABIC", "1180.SR": "Saudi National Bank", "1010.SR": "Riyad Bank",
    "2082.SR": "ACWA Power", "1150.SR": "Alinma Bank", "2280.SR": "Almarai",
    "1211.SR": "Ma'aden", "1060.SR": "Saudi Awwal Bank", "1050.SR": "Banque Saudi Fransi",
    "2020.SR": "SABIC Agri-Nutrients", "5110.SR": "Saudi Electricity",
    "4013.SR": "Dr. Sulaiman Al Habib", "4190.SR": "Jarir Marketplace",
}

def setup_big_data():
    print("🚀 STEP 1: Starting Multi-Source Data Ingestion...")

    # Bypass 403 Forbidden
    opener = urllib.request.build_opener()
    opener.addheaders = [('User-agent', 'Mozilla/5.0 (Windows NT 10.0; Win64; x64)')]
    urllib.request.install_opener(opener)

    # Access Wikipedia lists
    sp500_data = pd.read_html('https://en.wikipedia.org/wiki/List_of_S%26P_500_companies')[0]
    nasdaq_data = pd.read_html('https://en.wikipedia.org/wiki/Nasdaq-100')[4]

    sp500_col = 'Symbol' if 'Symbol' in sp500_data.columns else sp500_data.columns[0]
    nasdaq_col = 'Ticker' if 'Ticker' in nasdaq_data.columns else nasdaq_data.columns[0]

    raw_list = list(set(sp500_data[sp500_col].tolist() + nasdaq_data[nasdaq_col].tolist()))
    saudi_list = list(saudi_names.keys())

    all_tickers = [str(t).replace('.', '-') if '.SR' not in str(t) else t for t in raw_list + saudi_list]

    print(f"📡 Initial scan found {len(all_tickers)} potential entities.")
    # progress=False hides the messy loading bars to keep the notebook clean
    data = yf.download(all_tickers, period="1y", interval="1d", group_by='ticker', threads=True, progress=False)

    # --- DATA CLEANING PHASE ---
    df = data.stack(level=0).reset_index()
    df.columns = [df.columns[0], 'Ticker'] + list(df.columns[2:])

    final_count = df['Ticker'].nunique()
    saudi_count = df['Ticker'].str.endswith('.SR').sum() and df[df['Ticker'].str.endswith('.SR')]['Ticker'].nunique()

    print("\n" + "="*50)
    print("🛡️ AUTOMATED DATA INTEGRITY REPORT")
    print("="*50)
    print(f"✅ Success: Data is gathered from {final_count} companies.")
    print(f"🇸🇦 Saudi companies included: {saudi_count}")
    print(f"📊 Total Records: {len(df):,} individual data points.")
    print("-" * 50)
    print("SYSTEM NOTE: The engine automatically filtered out")
    print("invalid markers to ensure a high-fidelity dataset.")
    print("==================================================")

    return df, sp500_data

master_df, sector_info = setup_big_data()

🚀 STEP 1: Starting Multi-Source Data Ingestion...
📡 Initial scan found 522 potential entities.


HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: 10000"}}}
$10000: No data found, symbol may be delisted
HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: 30000"}}}
$30000: No data found, symbol may be delisted
$20000: No data found, symbol may be delisted

3 Failed downloads:
['10000', '30000', '20000']: No data found, symbol may be delisted



🛡️ AUTOMATED DATA INTEGRITY REPORT
✅ Success: Data is gathered from 519 companies.
🇸🇦 Saudi companies included: 16
📊 Total Records: 129,743 individual data points.
--------------------------------------------------
SYSTEM NOTE: The engine automatically filtered out
invalid markers to ensure a high-fidelity dataset.


In [0]:
#Cell 4
# 1. Show the shape (Rows, Columns)
print(f"TOTAL DATA POINTS COLLECTED: {master_df.size:,}")
print(f"NUMBER OF ROWS: {master_df.shape[0]:,}")
print(f"NUMBER OF COLUMNS: {master_df.shape[1]:,}")

# 2. Show the professional interactive table
display(master_df)

TOTAL DATA POINTS COLLECTED: 1,037,944
NUMBER OF ROWS: 129,743
NUMBER OF COLUMNS: 8


Date,Ticker,Open,High,Low,Close,Volume,Adj Close
2025-10-07T00:00:00.000Z,1010.SR,19.698443093842464,19.784026626120074,19.55580447796258,19.62712287902832,3783446.0,null
2025-10-07T00:00:00.000Z,1050.SR,17.839681498291576,17.88700088536143,17.574688237391207,17.754505157470703,1882891.0,null
2025-10-07T00:00:00.000Z,1060.SR,30.818347004842227,30.912768122970682,30.610624146757004,30.686161041259766,1240531.0,null
2025-10-07T00:00:00.000Z,1120.SR,68.7648812374961,68.89353936781056,68.12162003219726,68.63623046875,4768432.0,null
2025-10-07T00:00:00.000Z,1150.SR,21.70984416440265,21.725974920906694,21.564682122550337,21.645328521728516,4767547.0,null
2025-10-07T00:00:00.000Z,1180.SR,36.8173627201362,36.8173627201362,36.40134860592935,36.571537017822266,3691953.0,null
2025-10-07T00:00:00.000Z,1211.SR,67.9000015258789,68.0999984741211,66.6500015258789,66.6500015258789,1827602.0,null
2025-10-07T00:00:00.000Z,2010.SR,59.334434790469324,59.47752207830952,58.666683197021484,58.666683197021484,1644482.0,null
2025-10-07T00:00:00.000Z,2020.SR,113.47441542565336,115.27408730565094,113.28497133590675,114.23217010498047,811769.0,null
2025-10-07T00:00:00.000Z,2082.SR,210.30876811891036,216.39454393256673,207.51529001635174,214.4989776611328,1762873.0,null


In [0]:
#Cell 5
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px # Using px for the color sequence

# Create name_map from sector_info
name_map = dict(zip(sector_info['Symbol'], sector_info['Security']))
name_map.update(saudi_names)   # <-- NEW LINE: adds the Saudi company names

# 1. Prepare Data
results = []
for ticker in master_df['Ticker'].unique():
    s_df = master_df[master_df['Ticker'] == ticker].sort_values('Date')
    if len(s_df) < 10: continue
    growth = ((s_df['Close'].iloc[-1] - s_df['Close'].iloc[0]) / s_df['Close'].iloc[0]) * 100
    results.append({'Company': name_map.get(ticker, ticker), 'Growth': round(growth, 2)})

growth_df = pd.DataFrame(results).sort_values(by='Growth', ascending=False).reset_index(drop=True)

# 2. Create Dashboard
fig = make_subplots(rows=2, cols=1, row_heights=[0.35, 0.65],
                    specs=[[{"type": "table"}], [{"type": "scatter"}]],
                    vertical_spacing=0.08)

# Table (Top)
fig.add_trace(go.Table(
    header=dict(values=['Company', 'Growth % (TTM)'], fill_color='#1a237e', font=dict(color='white', size=13)),
    cells=dict(values=[growth_df.Company, growth_df.Growth], fill_color='#f5f5f5', align='left')), row=1, col=1)

# Interactive Circles (Bottom)
# We use a large color palette so every company looks different
colors = px.colors.qualitative.Alphabet + px.colors.qualitative.Dark24 + px.colors.qualitative.Light24
extended_colors = (colors * (len(growth_df) // len(colors) + 1))[:len(growth_df)]

fig.add_trace(go.Scatter(
    x=growth_df.index, # Using index for spacing
    y=growth_df['Growth'],
    mode='markers',
    marker=dict(
        size=12,
        color=extended_colors, # Each company gets a unique color from the palette
        line=dict(width=1, color='DarkSlateGrey')
    ),
    text=growth_df['Company'], # This tells plotly the name for the hover
    hovertemplate="<b>%{text}</b><br>Growth: %{y}%<extra></extra>",
), row=2, col=1)

# 3. Clean Formatting
fig.update_layout(
    height=850,
    title_text="<b>Audit 1: Capital Appreciation Analysis (Last 12 Months)</b>",
    template="plotly_white",
    showlegend=False,
    margin=dict(l=50, r=50, t=80, b=50)
)

# Hide EVERYTHING on the X-axis for that super neat look
fig.update_xaxes(
    showticklabels=False,
    showgrid=False,
    zeroline=False,
    title="", # Removed the "Hover for name" text
    rangeslider=dict(visible=True, thickness=0.05),
    range=[-1, 50], # Default view
    row=2, col=1
)

fig.update_yaxes(title="Growth %", gridcolor='#eeeeee', row=2, col=1)
fig.show()

In [0]:
#Cell 6
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px

# 1. Prepare Data
risk_results = []
for ticker in master_df['Ticker'].unique():
    s_df = master_df[master_df['Ticker'] == ticker]
    if len(s_df) < 10: continue
    # Calculation: Average daily spread (High-Low) relative to Open price
    risk = ((s_df['High'] - s_df['Low']) / s_df['Open']).mean() * 100
    risk_results.append({'Company': name_map.get(ticker, ticker), 'Risk': round(risk, 2)})

# Sorting by Risk (Lowest Risk/Safest companies first)
risk_df = pd.DataFrame(risk_results).sort_values(by='Risk', ascending=True).reset_index(drop=True)

# 2. Create Dashboard
fig = make_subplots(rows=2, cols=1, row_heights=[0.35, 0.65],
                    specs=[[{"type": "table"}], [{"type": "scatter"}]],
                    vertical_spacing=0.08)

# Table (Top)
fig.add_trace(go.Table(
    header=dict(values=['Company', 'Risk Score (Volatility)'], fill_color='#b71c1c', font=dict(color='white', size=13)),
    cells=dict(values=[risk_df.Company, risk_df.Risk], fill_color='#fff5f5', align='left')), row=1, col=1)

# Interactive Circles (Bottom) - Using a warmer "Risk" palette
risk_colors = px.colors.qualitative.Vivid + px.colors.qualitative.Pastel + px.colors.qualitative.Safe
extended_risk_colors = (risk_colors * (len(risk_df) // len(risk_colors) + 1))[:len(risk_df)]

fig.add_trace(go.Scatter(
    x=risk_df.index,
    y=risk_df['Risk'],
    mode='markers',
    marker=dict(
        size=12,
        color=extended_risk_colors,
        line=dict(width=1, color='DarkSlateGrey')
    ),
    text=risk_df['Company'],
    hovertemplate="<b>%{text}</b><br>Risk Score: %{y}<extra></extra>",
), row=2, col=1)

# 3. Clean Formatting
fig.update_layout(
    height=850,
    title_text="<b>Audit 2: Risk & Volatility Analysis (Last 12 Months)</b>",
    template="plotly_white",
    showlegend=False,
    margin=dict(l=50, r=50, t=80, b=50)
)

# Hide labels for the neat look
fig.update_xaxes(
    showticklabels=False,
    showgrid=False,
    zeroline=False,
    title="",
    rangeslider=dict(visible=True, thickness=0.05),
    range=[-1, 50],
    row=2, col=1
)

fig.update_yaxes(title="Volatility Score (Lower is Safer)", gridcolor='#eeeeee', row=2, col=1)
fig.show()
      

In [0]:
#Cell 7
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# 1. COMBINE AND FILTER
master_report = pd.merge(growth_df, risk_df, on='Company')

# Managerial Filter: Risk < 2.5 and Growth > 5%
shortlist_df = master_report[(master_report['Risk'] < 2.5) & (master_report['Growth'] > 5)].copy()

# Calculate Efficiency (The "Gain" metric)
shortlist_df['Efficiency'] = (shortlist_df['Growth'] / shortlist_df['Risk']).round(2)
top_10_candidates = shortlist_df.sort_values(by='Efficiency', ascending=False).head(10)

# 2. CREATE SUBPLOTS (Table on top, Bar Chart on bottom)
fig = make_subplots(
    rows=2, cols=1,
    row_heights=[0.4, 0.6],
    specs=[[{"type": "table"}], [{"type": "bar"}]],
    vertical_spacing=0.1
)

# ADD THE EFFICIENCY TABLE
fig.add_trace(
    go.Table(
        header=dict(
            values=['<b>Rank</b>', '<b>Company</b>', '<b>Growth %</b>', '<b>Risk Score</b>', '<b>Efficiency Score</b>'],
            fill_color='#1a237e',
            font=dict(color='white', size=12),
            align='center'
        ),
        cells=dict(
            values=[
                list(range(1, 11)),
                top_10_candidates['Company'],
                top_10_candidates['Growth'],
                top_10_candidates['Risk'],
                top_10_candidates['Efficiency']
            ],
            fill_color='#f5f5f5',
            align='center'
        )
    ), row=1, col=1
)

# ADD THE COMPARISON BAR CHART
fig.add_trace(go.Bar(
    x=top_10_candidates['Company'], y=top_10_candidates['Growth'],
    name='Yearly Growth %', marker_color='#2e7d32', text=top_10_candidates['Growth'], textposition='auto'
), row=2, col=1)

fig.add_trace(go.Bar(
    x=top_10_candidates['Company'], y=top_10_candidates['Risk'],
    name='Risk Score', marker_color='#c62828', text=top_10_candidates['Risk'], textposition='auto'
), row=2, col=1)

# 3. FORMATTING
fig.update_layout(
    height=800,
    title_text="<b>Step 3: The Managerial Shortlist (Efficiency & Performance Audit)</b>",
    template='plotly_white',
    barmode='group',
    showlegend=True,
    legend=dict(orientation="h", yanchor="bottom", y=0.58, xanchor="right", x=1)
)

fig.show()

In [0]:
#Cell 8
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# 1. CALCULATE MARKET AVERAGES FOR COMPARISON
avg_growth = master_report['Growth'].mean()
avg_risk = master_report['Risk'].mean()

# 2. GET THE TOP STOCK FROM THE SHORTLIST
winner = top_10_candidates.iloc[0]

# 3. CREATE A PERFORMANCE DASHBOARD
fig = make_subplots(
    rows=1, cols=3,
    specs=[[{"type": "indicator"}, {"type": "indicator"}, {"type": "indicator"}]],
    subplot_titles=("Growth vs Market Avg", "Risk vs Market Avg", "Efficiency Score")
)

# GROWTH
fig.add_trace(go.Indicator(
    mode="number+delta",
    value=winner['Growth'],
    delta={'reference': avg_growth, 'relative': True, 'valueformat': '.1%'},
    title={"text": "Yearly Growth"},
    domain={'x': [0, 0.3], 'y': [0, 1]}
), row=1, col=1)

# RISK (lower is better, so the colours are reversed)
fig.add_trace(go.Indicator(
    mode="number+delta",
    value=winner['Risk'],
    delta={'reference': avg_risk, 'relative': True, 'valueformat': '.1%',
           'increasing': {'color': "red"}, 'decreasing': {'color': "green"}},
    title={"text": "Risk Score"},
    domain={'x': [0.35, 0.65], 'y': [0, 1]}
), row=1, col=2)

# EFFICIENCY
fig.add_trace(go.Indicator(
    mode="number",
    value=winner['Efficiency'],
    title={"text": "Efficiency Score"},
    domain={'x': [0.7, 1], 'y': [0, 1]}
), row=1, col=3)

# 4. TITLE
fig.update_layout(
    height=400,
    title={
        'text': f"<b>TOP PICK: {winner['Company'].upper()}</b><br><span style='font-size:14px'>Highest efficiency among the stocks that fit the risk and growth limits</span>",
        'y': 0.9, 'x': 0.5, 'xanchor': 'center', 'yanchor': 'top'
    },
    template='plotly_white'
)
fig.show()

print(f"CONCLUSION: {winner['Company']} beats the market average growth by {((winner['Growth']/avg_growth)-1)*100:.1f}%")
print(f"while its risk is {((1-winner['Risk']/avg_risk))*100:.1f}% lower than the market average.")


CONCLUSION: Warner Bros. Discovery beats the market average growth by 235.6%
while its risk is 36.0% lower than the market average.


In [0]:
#Cell 9
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ---- YOUR SETTINGS: how much risk and how little growth you will accept ----
max_risk = 2.5      # change this: lower = safer stocks only
min_growth = 5      # change this: minimum 12-month growth (%)
# ----------------------------------------------------------------------------

# 1. Saudi companies only, with growth, risk and efficiency
saudi_report = master_report[master_report['Company'].isin(saudi_names.values())].copy()
saudi_report['Efficiency'] = (saudi_report['Growth'] / saudi_report['Risk']).round(2)
saudi_report['Fits my risk limit'] = ((saudi_report['Risk'] < max_risk) &
                                      (saudi_report['Growth'] > min_growth)).map({True: 'Yes', False: 'No'})

# 2. Rank by efficiency (best first)
saudi_report = saudi_report.sort_values(by='Efficiency', ascending=False).reset_index(drop=True)

if saudi_report.empty:
    print("No Saudi companies found. Check that Cell 3 and Cell 5 ran after the changes.")
else:
    fig = make_subplots(
        rows=2, cols=1, row_heights=[0.45, 0.55],
        specs=[[{"type": "table"}], [{"type": "bar"}]],
        vertical_spacing=0.1
    )

    # Table
    fig.add_trace(go.Table(
        header=dict(
            values=['<b>Rank</b>', '<b>Company</b>', '<b>Growth %</b>', '<b>Risk Score</b>',
                    '<b>Efficiency Score</b>', f'<b>Fits limit (risk &lt; {max_risk}, growth &gt; {min_growth}%)</b>'],
            fill_color='#1a237e', font=dict(color='white', size=12), align='center'),
        cells=dict(
            values=[list(range(1, len(saudi_report) + 1)), saudi_report['Company'],
                    saudi_report['Growth'], saudi_report['Risk'],
                    saudi_report['Efficiency'], saudi_report['Fits my risk limit']],
            fill_color='#f5f5f5', align='center')
    ), row=1, col=1)

    # Bar chart: growth vs risk for each Saudi company
    fig.add_trace(go.Bar(x=saudi_report['Company'], y=saudi_report['Growth'],
                         name='Yearly Growth %', marker_color='#2e7d32'), row=2, col=1)
    fig.add_trace(go.Bar(x=saudi_report['Company'], y=saudi_report['Risk'],
                         name='Risk Score', marker_color='#c62828'), row=2, col=1)

    fig.update_layout(
        height=850, template='plotly_white', barmode='group', showlegend=True,
        title_text="<b>Audit 4: Efficiency Analysis of Saudi Companies (Last 12 Months)</b>"
    )
    fig.show()

    # 3. Short summary
    fits = saudi_report[saudi_report['Fits my risk limit'] == 'Yes']
    print(f"Saudi companies analysed: {len(saudi_report)}")
    print(f"Companies that fit my limit (risk < {max_risk}, growth > {min_growth}%): {len(fits)}")
    if not fits.empty:
        best = fits.iloc[0]
        print(f"Best choice within my limit: {best['Company']} "
              f"(growth {best['Growth']}%, risk {best['Risk']}, efficiency {best['Efficiency']})")
    else:
        print("No Saudi company fits this limit. Try raising max_risk or lowering min_growth.")

Saudi companies analysed: 16
Companies that fit my limit (risk < 2.5, growth > 5%): 6
Best choice within my limit: Jarir Marketplace (growth 27.03%, risk 1.57, efficiency 17.22)
